In [ ]:
"""
Consumer Expenditure Survey 2024 - Quarterly Analysis
Comprehensive quarter-by-quarter comparison and trend analysis
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style("whitegrid")
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10

# ============================================================================
# CONFIGURATION
# ============================================================================

data_dir = Path('./data/intrvw24')
output_dir = Path('./data/output-quarterly')
output_dir.mkdir(parents=True, exist_ok=True)

quarters_2024 = ['242', '243', '244']  # Q2, Q3, Q4 2024
quarter_labels = {
    '242': 'Q2 2024\n(Apr-Jun)',
    '243': 'Q3 2024\n(Jul-Sep)',
    '244': 'Q4 2024\n(Oct-Dec)'
}

print("="*80)
print("QUARTERLY ANALYSIS - CONSUMER EXPENDITURE SURVEY 2024")
print("="*80)
print(f"Analyzing: Q2, Q3, Q4 2024")
print(f"Output directory: {output_dir}")
print("="*80)

# ============================================================================
# SECTION 1: DATA LOADING
# ============================================================================

print("\n" + "="*80)
print("SECTION 1: LOADING QUARTERLY DATA")
print("="*80)

quarterly_data = {}

for quarter in quarters_2024:
    file_path = data_dir / f'fmli{quarter}.csv'
    if file_path.exists():
        df = pd.read_csv(file_path)
        df['quarter'] = quarter
        df['quarter_label'] = quarter_labels[quarter]
        quarterly_data[quarter] = df
        print(f"✓ Loaded Q{quarter[-1]} 2024: {len(df):,} records, {len(df.columns)} columns")
    else:
        print(f"✗ File not found: {file_path}")

if len(quarterly_data) == 0:
    print("ERROR: No data files loaded.")
    exit(1)

# Combine all quarters
fmli = pd.concat(quarterly_data.values(), ignore_index=True)
print(f"\n✓ Combined dataset: {len(fmli):,} total records")

# ============================================================================
# SECTION 1.5: MISSING VALUES HANDLING
# Strategy: expenditure missing = $0, income = median, categorical = "MISSING"
# ============================================================================

print("\n" + "="*80)
print("SECTION 1.5: MISSING VALUES HANDLING")
print("="*80)

fmli_clean = fmli.copy()

# 1. Drop columns with >95% missing
cols_to_drop = (fmli_clean.isnull().sum() / len(fmli_clean)) > 0.95
dropped_cols = cols_to_drop[cols_to_drop].index.tolist()
fmli_clean = fmli_clean.drop(columns=dropped_cols)
print(f"Dropped {len(dropped_cols)} columns with >95% missing")

# 2. Expenditure variables: fill with 0
expend_patterns = ['PQ', 'CQ', 'EXPPQ', 'EXPCQ', 'FD', 'HOUS', 'TRAN', 'HEALTH', 'ENTERT', 'EDUC', 'APPAR']
expend_cols = [c for c in fmli_clean.columns if any(p in c for p in expend_patterns) and fmli_clean[c].dtype in ['float64', 'int64']]
fmli_clean[expend_cols] = fmli_clean[expend_cols].fillna(0)

# 3. Income variables: median imputation
income_keywords = ['INC', 'INCOME', 'FINC', 'SALARY', 'WAGE', 'RETIR', 'INVEST', 'RENT', 'TAX']
income_cols = [c for c in fmli_clean.columns if any(k in c for k in income_keywords) and c not in expend_cols 
               and fmli_clean[c].dtype in ['float64', 'int64']]
for col in income_cols:
    if fmli_clean[col].isnull().any():
        fmli_clean[col] = fmli_clean[col].fillna(fmli_clean[col].median())

# 4. Other numerical: median imputation
for col in fmli_clean.select_dtypes(include=['float64', 'int64']).columns:
    if fmli_clean[col].isnull().any():
        fmli_clean[col] = fmli_clean[col].fillna(fmli_clean[col].median())

# 5. Categorical: fill with "MISSING"
for col in fmli_clean.select_dtypes(include=['object']).columns:
    if fmli_clean[col].isnull().any():
        fmli_clean[col] = fmli_clean[col].fillna('MISSING').astype(str)

fmli = fmli_clean
# Update quarterly_data to use cleaned data for consistency
for quarter in quarters_2024:
    quarterly_data[quarter] = fmli[fmli['quarter'] == quarter].copy()

remaining = fmli.isnull().sum().sum()
print(f"✓ Missing values handled. Remaining: {remaining}")

# ============================================================================
# SECTION 2: QUARTERLY SAMPLE SIZE COMPARISON
# ============================================================================

print("\n" + "="*80)
print("SECTION 2: SAMPLE SIZE ANALYSIS BY QUARTER")
print("="*80)

sample_size_summary = []

for quarter in quarters_2024:
    qdata = quarterly_data[quarter]
    
    summary = {
        'Quarter': quarter_labels[quarter].replace('\n', ' '),
        'Total_Records': len(qdata),
        'Unique_NEWID': qdata['NEWID'].nunique() if 'NEWID' in qdata.columns else 'N/A',
        'Variables': len(qdata.columns)
    }
    
    sample_size_summary.append(summary)
    
    print(f"\n{quarter_labels[quarter].replace(chr(10), ' ')}:")
    print(f"  Total records: {len(qdata):,}")
    if 'NEWID' in qdata.columns:
        print(f"  Unique consumer units: {qdata['NEWID'].nunique():,}")
    print(f"  Variables: {len(qdata.columns)}")

sample_size_df = pd.DataFrame(sample_size_summary)
sample_size_df.to_csv(output_dir / '01_quarterly_sample_sizes.csv', index=False)
print(f"\n✓ Saved: 01_quarterly_sample_sizes.csv")

# Visualization: Sample sizes
fig, ax = plt.subplots(figsize=(10, 6))
quarters_short = [quarter_labels[q] for q in quarters_2024]
sample_counts = [len(quarterly_data[q]) for q in quarters_2024]

colors = ['#3498db', '#2ecc71', '#e74c3c']
bars = ax.bar(range(len(quarters_short)), sample_counts, color=colors, 
              edgecolor='black', alpha=0.8, width=0.6)
ax.set_xticks(range(len(quarters_short)))
ax.set_xticklabels(quarters_short)
ax.set_ylabel('Number of Records')
ax.set_title('Sample Size by Quarter - 2024', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# Add value labels
for bar, count in zip(bars, sample_counts):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
           f'{count:,}', ha='center', va='bottom', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.savefig(output_dir / '01_viz_sample_sizes.png', dpi=300, bbox_inches='tight')
plt.close()
print("✓ Saved: 01_viz_sample_sizes.png")

# ============================================================================
# SECTION 3: DEMOGRAPHIC COMPARISON BY QUARTER
# ============================================================================

print("\n" + "="*80)
print("SECTION 3: DEMOGRAPHIC COMPARISON BY QUARTER")
print("="*80)

# Age comparison
if 'AGE_REF' in fmli.columns:
    age_by_quarter = []
    
    for quarter in quarters_2024:
        qdata = quarterly_data[quarter]
        age_data = qdata['AGE_REF'].dropna()
        
        age_by_quarter.append({
            'Quarter': quarter_labels[quarter].replace('\n', ' '),
            'Mean_Age': age_data.mean(),
            'Median_Age': age_data.median(),
            'Std_Dev': age_data.std(),
            'Min': age_data.min(),
            'Max': age_data.max()
        })
    
    age_comparison_df = pd.DataFrame(age_by_quarter)
    age_comparison_df.to_csv(output_dir / '02_age_by_quarter.csv', index=False)
    print("\n✓ Saved: 02_age_by_quarter.csv")
    
    print("\nAGE STATISTICS BY QUARTER:")
    print(age_comparison_df.to_string(index=False))
    
    # Visualization: Age distribution by quarter
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # Box plots
    age_data_list = [quarterly_data[q]['AGE_REF'].dropna() for q in quarters_2024]
    labels = [quarter_labels[q] for q in quarters_2024]
    
    bp = axes[0, 0].boxplot(age_data_list, labels=labels, patch_artist=True)
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    axes[0, 0].set_ylabel('Age')
    axes[0, 0].set_title('Age Distribution by Quarter', fontweight='bold')
    axes[0, 0].grid(axis='y', alpha=0.3)
    
    # Mean comparison
    means = [age_data.mean() for age_data in age_data_list]
    bars = axes[0, 1].bar(range(len(labels)), means, color=colors, 
                          edgecolor='black', alpha=0.8)
    axes[0, 1].set_xticks(range(len(labels)))
    axes[0, 1].set_xticklabels(labels)
    axes[0, 1].set_ylabel('Average Age')
    axes[0, 1].set_title('Average Age by Quarter', fontweight='bold')
    axes[0, 1].grid(axis='y', alpha=0.3)
    
    for bar, mean_val in zip(bars, means):
        height = bar.get_height()
        axes[0, 1].text(bar.get_x() + bar.get_width()/2., height,
                       f'{mean_val:.1f}', ha='center', va='bottom', fontweight='bold')
    
    # Overlapping histograms
    for i, (quarter, color) in enumerate(zip(quarters_2024, colors)):
        age_data = quarterly_data[quarter]['AGE_REF'].dropna()
        axes[1, 0].hist(age_data, bins=30, alpha=0.5, label=quarter_labels[quarter],
                       color=color, edgecolor='black')
    axes[1, 0].set_xlabel('Age')
    axes[1, 0].set_ylabel('Frequency')
    axes[1, 0].set_title('Age Distribution - All Quarters Overlaid', fontweight='bold')
    axes[1, 0].legend()
    axes[1, 0].grid(alpha=0.3)
    
    # Violin plots
    positions = range(len(labels))
    parts = axes[1, 1].violinplot(age_data_list, positions=positions, showmeans=True, 
                                   showmedians=True)
    for i, pc in enumerate(parts['bodies']):
        pc.set_facecolor(colors[i])
        pc.set_alpha(0.7)
    axes[1, 1].set_xticks(positions)
    axes[1, 1].set_xticklabels(labels)
    axes[1, 1].set_ylabel('Age')
    axes[1, 1].set_title('Age Distribution - Violin Plots', fontweight='bold')
    axes[1, 1].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(output_dir / '02_viz_age_by_quarter.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("✓ Saved: 02_viz_age_by_quarter.png")

# Family Size comparison
if 'FAM_SIZE' in fmli.columns:
    print("\nFAMILY SIZE DISTRIBUTION BY QUARTER:")
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    x = np.arange(fmli['FAM_SIZE'].max() + 1)
    width = 0.25
    
    for i, quarter in enumerate(quarters_2024):
        qdata = quarterly_data[quarter]
        fam_counts = qdata['FAM_SIZE'].value_counts().sort_index()
        fam_pct = (fam_counts / len(qdata) * 100).reindex(x, fill_value=0)
        
        offset = width * (i - 1)
        ax.bar(x + offset, fam_pct.values, width, label=quarter_labels[quarter],
               color=colors[i], edgecolor='black', alpha=0.8)
    
    ax.set_xlabel('Family Size')
    ax.set_ylabel('Percentage of Consumer Units (%)')
    ax.set_title('Family Size Distribution by Quarter', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(output_dir / '03_viz_family_size_by_quarter.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("✓ Saved: 03_viz_family_size_by_quarter.png")

# Regional Distribution
if 'REGION' in fmli.columns:
    region_labels_map = {1: 'Northeast', 2: 'Midwest', 3: 'South', 4: 'West'}
    
    regional_comparison = []
    for quarter in quarters_2024:
        qdata = quarterly_data[quarter]
        region_counts = qdata['REGION'].value_counts()
        total = len(qdata)
        
        for region_code, count in region_counts.items():
            regional_comparison.append({
                'Quarter': quarter_labels[quarter].replace('\n', ' '),
                'Region': region_labels_map.get(region_code, f'Region {region_code}'),
                'Count': count,
                'Percentage': (count / total * 100)
            })
    
    regional_df = pd.DataFrame(regional_comparison)
    regional_df.to_csv(output_dir / '03_regional_distribution_by_quarter.csv', index=False)
    print("✓ Saved: 03_regional_distribution_by_quarter.csv")
    
    # Visualization
    fig, ax = plt.subplots(figsize=(12, 6))
    
    pivot_data = regional_df.pivot(index='Region', columns='Quarter', values='Percentage')
    pivot_data.plot(kind='bar', ax=ax, color=colors, edgecolor='black', alpha=0.8)
    
    ax.set_ylabel('Percentage of Sample (%)')
    ax.set_title('Regional Distribution by Quarter', fontsize=14, fontweight='bold')
    ax.legend(title='Quarter')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
    ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(output_dir / '03_viz_regional_distribution.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("✓ Saved: 03_viz_regional_distribution.png")

# =============================================
# SECTION 4: INCOME ANALYSIS BY QUARTER
# =============================================

print("\n" + "="*80)
print("SECTION 4: INCOME ANALYSIS BY QUARTER")
print("="*80)

if 'FINCBTXM' in fmli.columns:
    income_by_quarter = []
    
    for quarter in quarters_2024:
        qdata = quarterly_data[quarter]
        income_data = qdata['FINCBTXM'].dropna()
        income_data = income_data[income_data > 0]
        
        income_by_quarter.append({
            'Quarter': quarter_labels[quarter].replace('\n', ' '),
            'Mean_Income': income_data.mean(),
            'Median_Income': income_data.median(),
            'Std_Dev': income_data.std(),
            'Q1_25th': income_data.quantile(0.25),
            'Q3_75th': income_data.quantile(0.75),
            'Min': income_data.min(),
            'Max': income_data.max(),
            'N': len(income_data)
        })
    
    income_comparison_df = pd.DataFrame(income_by_quarter)
    income_comparison_df.to_csv(output_dir / '04_income_by_quarter.csv', index=False)
    print("\n✓ Saved: 04_income_by_quarter.csv")
    
    print("\nINCOME STATISTICS BY QUARTER:")
    for _, row in income_comparison_df.iterrows():
        print(f"\n{row['Quarter']}:")
        print(f"  Mean: ${row['Mean_Income']:,.2f}")
        print(f"  Median: ${row['Median_Income']:,.2f}")
        print(f"  Std Dev: ${row['Std_Dev']:,.2f}")
    
    # Comprehensive income visualizations
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # Box plots
    income_data_list = []
    for quarter in quarters_2024:
        income_data = quarterly_data[quarter]['FINCBTXM'].dropna()
        income_data = income_data[income_data > 0]
        income_data_list.append(income_data)
    
    labels = [quarter_labels[q] for q in quarters_2024]
    
    bp = axes[0, 0].boxplot(income_data_list, labels=labels, patch_artist=True)
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    axes[0, 0].set_ylabel('Income Before Taxes ($)')
    axes[0, 0].set_title('Income Distribution by Quarter', fontweight='bold')
    axes[0, 0].grid(axis='y', alpha=0.3)
    axes[0, 0].ticklabel_format(style='plain', axis='y')
    
    # Mean and Median comparison
    means = [data.mean() for data in income_data_list]
    medians = [data.median() for data in income_data_list]
    
    x_pos = np.arange(len(labels))
    width = 0.35
    
    bars1 = axes[0, 1].bar(x_pos - width/2, means, width, label='Mean',
                           color='#3498db', edgecolor='black', alpha=0.8)
    bars2 = axes[0, 1].bar(x_pos + width/2, medians, width, label='Median',
                           color='#2ecc71', edgecolor='black', alpha=0.8)
    
    axes[0, 1].set_xticks(x_pos)
    axes[0, 1].set_xticklabels(labels)
    axes[0, 1].set_ylabel('Income ($)')
    axes[0, 1].set_title('Mean vs Median Income by Quarter', fontweight='bold')
    axes[0, 1].legend()
    axes[0, 1].grid(axis='y', alpha=0.3)
    axes[0, 1].ticklabel_format(style='plain', axis='y')
    
    # Add value labels
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            axes[0, 1].text(bar.get_x() + bar.get_width()/2., height,
                          f'${height/1000:.0f}K', ha='center', va='bottom', fontsize=9)
    
    # Violin plots
    positions = range(len(labels))
    parts = axes[1, 0].violinplot(income_data_list, positions=positions, 
                                   showmeans=True, showmedians=True)
    for i, pc in enumerate(parts['bodies']):
        pc.set_facecolor(colors[i])
        pc.set_alpha(0.7)
    axes[1, 0].set_xticks(positions)
    axes[1, 0].set_xticklabels(labels)
    axes[1, 0].set_ylabel('Income ($)')
    axes[1, 0].set_title('Income Distribution - Violin Plots', fontweight='bold')
    axes[1, 0].grid(axis='y', alpha=0.3)
    axes[1, 0].ticklabel_format(style='plain', axis='y')
    
    # Trend line
    means_series = pd.Series(means, index=range(len(means)))
    axes[1, 1].plot(range(len(labels)), means, marker='o', linewidth=2, 
                    markersize=10, color='#e74c3c')
    axes[1, 1].set_xticks(range(len(labels)))
    axes[1, 1].set_xticklabels(labels)
    axes[1, 1].set_ylabel('Average Income ($)')
    axes[1, 1].set_title('Income Trend Across Quarters', fontweight='bold')
    axes[1, 1].grid(alpha=0.3)
    axes[1, 1].ticklabel_format(style='plain', axis='y')
    
    # Add value labels on trend line
    for i, (x, y) in enumerate(zip(range(len(labels)), means)):
        axes[1, 1].text(x, y, f'${y/1000:.0f}K', ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(output_dir / '04_viz_income_by_quarter.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("✓ Saved: 04_viz_income_by_quarter.png")

# ===============================================
# SECTION 5: EXPENDITURE ANALYSIS BY QUARTER
# ===============================================

print("\n" + "="*80)
print("SECTION 5: EXPENDITURE ANALYSIS BY QUARTER")
print("="*80)

if 'TOTEXPPQ' in fmli.columns:
    expend_by_quarter = []
    
    for quarter in quarters_2024:
        qdata = quarterly_data[quarter]
        expend_data = qdata['TOTEXPPQ'].dropna()
        expend_data = expend_data[expend_data > 0]
        
        expend_by_quarter.append({
            'Quarter': quarter_labels[quarter].replace('\n', ' '),
            'Mean_Expenditure': expend_data.mean(),
            'Median_Expenditure': expend_data.median(),
            'Std_Dev': expend_data.std(),
            'Q1_25th': expend_data.quantile(0.25),
            'Q3_75th': expend_data.quantile(0.75),
            'Min': expend_data.min(),
            'Max': expend_data.max(),
            'N': len(expend_data)
        })
    
    expend_comparison_df = pd.DataFrame(expend_by_quarter)
    expend_comparison_df.to_csv(output_dir / '05_expenditure_by_quarter.csv', index=False)
    print("\n✓ Saved: 05_expenditure_by_quarter.csv")
    
    print("\nEXPENDITURE STATISTICS BY QUARTER:")
    for _, row in expend_comparison_df.iterrows():
        print(f"\n{row['Quarter']}:")
        print(f"  Mean: ${row['Mean_Expenditure']:,.2f}")
        print(f"  Median: ${row['Median_Expenditure']:,.2f}")
        print(f"  Std Dev: ${row['Std_Dev']:,.2f}")
    
    # Expenditure visualizations
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # Box plots
    expend_data_list = []
    for quarter in quarters_2024:
        expend_data = quarterly_data[quarter]['TOTEXPPQ'].dropna()
        expend_data = expend_data[expend_data > 0]
        expend_data_list.append(expend_data)
    
    labels = [quarter_labels[q] for q in quarters_2024]
    
    bp = axes[0, 0].boxplot(expend_data_list, labels=labels, patch_artist=True)
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    axes[0, 0].set_ylabel('Total Expenditure ($)')
    axes[0, 0].set_title('Expenditure Distribution by Quarter', fontweight='bold')
    axes[0, 0].grid(axis='y', alpha=0.3)
    axes[0, 0].ticklabel_format(style='plain', axis='y')
    
    # Mean comparison
    means = [data.mean() for data in expend_data_list]
    bars = axes[0, 1].bar(range(len(labels)), means, color=colors, 
                          edgecolor='black', alpha=0.8)
    axes[0, 1].set_xticks(range(len(labels)))
    axes[0, 1].set_xticklabels(labels)
    axes[0, 1].set_ylabel('Average Expenditure ($)')
    axes[0, 1].set_title('Average Total Expenditure by Quarter', fontweight='bold')
    axes[0, 1].grid(axis='y', alpha=0.3)
    axes[0, 1].ticklabel_format(style='plain', axis='y')
    
    for bar, mean_val in zip(bars, means):
        height = bar.get_height()
        axes[0, 1].text(bar.get_x() + bar.get_width()/2., height,
                       f'${mean_val/1000:.1f}K', ha='center', va='bottom', fontweight='bold')
    
    # Overlapping histograms
    for i, (quarter, color) in enumerate(zip(quarters_2024, colors)):
        expend_data = quarterly_data[quarter]['TOTEXPPQ'].dropna()
        expend_data = expend_data[expend_data > 0]
        axes[1, 0].hist(expend_data, bins=50, alpha=0.5, label=quarter_labels[quarter],
                       color=color, edgecolor='black')
    axes[1, 0].set_xlabel('Total Expenditure ($)')
    axes[1, 0].set_ylabel('Frequency')
    axes[1, 0].set_title('Expenditure Distribution - All Quarters Overlaid', fontweight='bold')
    axes[1, 0].legend()
    axes[1, 0].grid(alpha=0.3)
    axes[1, 0].ticklabel_format(style='plain', axis='x')
    
    # Trend line
    axes[1, 1].plot(range(len(labels)), means, marker='o', linewidth=2, 
                    markersize=10, color='#9b59b6')
    axes[1, 1].set_xticks(range(len(labels)))
    axes[1, 1].set_xticklabels(labels)
    axes[1, 1].set_ylabel('Average Expenditure ($)')
    axes[1, 1].set_title('Expenditure Trend Across Quarters', fontweight='bold')
    axes[1, 1].grid(alpha=0.3)
    axes[1, 1].ticklabel_format(style='plain', axis='y')
    
    for i, (x, y) in enumerate(zip(range(len(labels)), means)):
        axes[1, 1].text(x, y, f'${y/1000:.1f}K', ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(output_dir / '05_viz_expenditure_by_quarter.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("✓ Saved: 05_viz_expenditure_by_quarter.png")

# ==================================================
# SECTION 6: EXPENDITURE CATEGORIES BY QUARTER
# ==================================================

print("\n" + "="*80)
print("SECTION 6: EXPENDITURE CATEGORIES BY QUARTER")
print("="*80)

expenditure_vars = {
    'FOODPQ': 'Food',
    'HOUSPQ': 'Housing',
    'TRANSPQ': 'Transportation',
    'HEALTHPQ': 'Healthcare',
    'ENTERTPQ': 'Entertainment',
    'APPARPQ': 'Apparel',
    'EDUCAPQ': 'Education'
}

available_expend = {k: v for k, v in expenditure_vars.items() if k in fmli.columns}

if available_expend:
    category_quarterly_data = []
    
    for quarter in quarters_2024:
        qdata = quarterly_data[quarter]
        
        for var, name in available_expend.items():
            data = qdata[var].dropna()
            data = data[data > 0]
            
            if len(data) > 0:
                category_quarterly_data.append({
                    'Quarter': quarter_labels[quarter].replace('\n', ' '),
                    'Category': name,
                    'Mean': data.mean(),
                    'Median': data.median(),
                    'Std_Dev': data.std(),
                    'N': len(data)
                })
    
    category_df = pd.DataFrame(category_quarterly_data)
    category_df.to_csv(output_dir / '06_categories_by_quarter.csv', index=False)
    print("\n✓ Saved: 06_categories_by_quarter.csv")
    
    # Visualization 1: Grouped bar chart
    fig, ax = plt.subplots(figsize=(14, 8))
    
    pivot_data = category_df.pivot(index='Category', columns='Quarter', values='Mean')
    pivot_data = pivot_data.sort_values(by=pivot_data.columns[0], ascending=False)
    
    pivot_data.plot(kind='bar', ax=ax, color=colors, edgecolor='black', alpha=0.8, width=0.8)
    
    ax.set_ylabel('Average Expenditure ($)')
    ax.set_xlabel('')
    ax.set_title('Average Expenditure by Category and Quarter', fontsize=14, fontweight='bold')
    ax.legend(title='Quarter', loc='upper right')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
    ax.grid(axis='y', alpha=0.3)
    ax.ticklabel_format(style='plain', axis='y')
    
    plt.tight_layout()
    plt.savefig(output_dir / '06_viz_categories_by_quarter.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("✓ Saved: 06_viz_categories_by_quarter.png")
    
    # Visualization 2: Stacked area chart showing trends
    fig, ax = plt.subplots(figsize=(12, 7))
    
    # Prepare data for stacked area
    quarters_for_plot = [quarter_labels[q].replace('\n', ' ') for q in quarters_2024]
    
    categories_means = {}
    for category in available_expend.values():
        cat_data = category_df[category_df['Category'] == category].sort_values('Quarter')
        categories_means[category] = cat_data['Mean'].values
    
    # Create stacked area plot
    ax.stackplot(range(len(quarters_for_plot)), 
                 *categories_means.values(),
                 labels=categories_means.keys(),
                 alpha=0.8)
    
    ax.set_xticks(range(len(quarters_for_plot)))
    ax.set_xticklabels(quarters_for_plot)
    ax.set_ylabel('Cumulative Average Expenditure ($)')
    ax.set_title('Expenditure Composition by Quarter (Stacked)', fontsize=14, fontweight='bold')
    ax.legend(loc='upper left', bbox_to_anchor=(1, 1))
    ax.grid(alpha=0.3)
    ax.ticklabel_format(style='plain', axis='y')
    
    plt.tight_layout()
    plt.savefig(output_dir / '06_viz_categories_stacked.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("✓ Saved: 06_viz_categories_stacked.png")
    
    # Visualization 3: Individual category trends
    n_categories = len(available_expend)
    n_cols = 2
    n_rows = (n_categories + 1) // 2
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4*n_rows))
    axes = axes.flatten()
    
    for idx, category in enumerate(available_expend.values()):
        cat_data = category_df[category_df['Category'] == category].sort_values('Quarter')
        
        axes[idx].plot(range(len(quarters_for_plot)), cat_data['Mean'].values, 
                      marker='o', linewidth=2, markersize=8, color=colors[0])
        axes[idx].set_xticks(range(len(quarters_for_plot)))
        axes[idx].set_xticklabels(quarters_for_plot, rotation=45, ha='right')
        axes[idx].set_ylabel('Average ($)')
        axes[idx].set_title(f'{category} Trend', fontweight='bold')
        axes[idx].grid(alpha=0.3)
        axes[idx].ticklabel_format(style='plain', axis='y')
        
        # value labels
        for x, y in enumerate(cat_data['Mean'].values):
            axes[idx].text(x, y, f'${y:,.0f}', ha='center', va='bottom', fontsize=9)
    
    # Hiding unused subplots
    for idx in range(len(available_expend), len(axes)):
        axes[idx].set_visible(False)
    
    plt.tight_layout()
    plt.savefig(output_dir / '06_viz_individual_category_trends.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("✓ Saved: 06_viz_individual_category_trends.png")

# =================================================
# SECTION 7: INCOME VS EXPENDITURE BY QUARTER
# =================================================

print("\n" + "="*80)
print("SECTION 7: INCOME VS EXPENDITURE RELATIONSHIP BY QUARTER")
print("="*80)

if 'FINCBTXM' in fmli.columns and 'TOTEXPPQ' in fmli.columns:
    income_expend_quarterly = []
    
    for quarter in quarters_2024:
        qdata = quarterly_data[quarter]
        analysis_df = qdata[['FINCBTXM', 'TOTEXPPQ']].dropna()
        analysis_df = analysis_df[(analysis_df['FINCBTXM'] > 0) & (analysis_df['TOTEXPPQ'] > 0)]
        
        if len(analysis_df) > 0:
            correlation = analysis_df['FINCBTXM'].corr(analysis_df['TOTEXPPQ'])
            expend_income_ratio = (analysis_df['TOTEXPPQ'] / analysis_df['FINCBTXM']).mean()
            savings_rate = 1 - expend_income_ratio
            
            income_expend_quarterly.append({
                'Quarter': quarter_labels[quarter].replace('\n', ' '),
                'Avg_Income': analysis_df['FINCBTXM'].mean(),
                'Avg_Expenditure': analysis_df['TOTEXPPQ'].mean(),
                'Correlation': correlation,
                'Expend_Income_Ratio': expend_income_ratio,
                'Savings_Rate': savings_rate,
                'N': len(analysis_df)
            })
    
    income_expend_df = pd.DataFrame(income_expend_quarterly)
    income_expend_df.to_csv(output_dir / '07_income_vs_expenditure_by_quarter.csv', index=False)
    print("\n✓ Saved: 07_income_vs_expenditure_by_quarter.csv")
    
    print("\nINCOME VS EXPENDITURE BY QUARTER:")
    for _, row in income_expend_df.iterrows():
        print(f"\n{row['Quarter']}:")
        print(f"  Avg Income: ${row['Avg_Income']:,.2f}")
        print(f"  Avg Expenditure: ${row['Avg_Expenditure']:,.2f}")
        print(f"  Correlation: {row['Correlation']:.3f}")
        print(f"  Expend/Income Ratio: {row['Expend_Income_Ratio']:.3f}")
        print(f"  Savings Rate: {row['Savings_Rate']*100:.1f}%")
    
    # Visualization
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    quarters_for_plot = [quarter_labels[q].replace('\n', ' ') for q in quarters_2024]
    
    # Income vs Expenditure comparison
    x_pos = np.arange(len(quarters_for_plot))
    width = 0.35
    
    bars1 = axes[0, 0].bar(x_pos - width/2, income_expend_df['Avg_Income'], width, 
                           label='Income', color='#2ecc71', edgecolor='black', alpha=0.8)
    bars2 = axes[0, 0].bar(x_pos + width/2, income_expend_df['Avg_Expenditure'], width,
                           label='Expenditure', color='#e74c3c', edgecolor='black', alpha=0.8)
    
    axes[0, 0].set_xticks(x_pos)
    axes[0, 0].set_xticklabels(quarters_for_plot)
    axes[0, 0].set_ylabel('Average Amount ($)')
    axes[0, 0].set_title('Income vs Expenditure by Quarter', fontweight='bold')
    axes[0, 0].legend()
    axes[0, 0].grid(axis='y', alpha=0.3)
    axes[0, 0].ticklabel_format(style='plain', axis='y')
    
    # Correlation trend
    axes[0, 1].plot(range(len(quarters_for_plot)), income_expend_df['Correlation'], 
                    marker='o', linewidth=2, markersize=10, color='#3498db')
    axes[0, 1].set_xticks(range(len(quarters_for_plot)))
    axes[0, 1].set_xticklabels(quarters_for_plot)
    axes[0, 1].set_ylabel('Correlation Coefficient')
    axes[0, 1].set_title('Income-Expenditure Correlation by Quarter', fontweight='bold')
    axes[0, 1].grid(alpha=0.3)
    axes[0, 1].set_ylim(0, 1)
    
    for x, y in enumerate(income_expend_df['Correlation']):
        axes[0, 1].text(x, y, f'{y:.3f}', ha='center', va='bottom', fontweight='bold')
    
    # Savings rate trend
    savings_pct = income_expend_df['Savings_Rate'] * 100
    bars = axes[1, 0].bar(range(len(quarters_for_plot)), savings_pct, 
                          color='#9b59b6', edgecolor='black', alpha=0.8)
    axes[1, 0].set_xticks(range(len(quarters_for_plot)))
    axes[1, 0].set_xticklabels(quarters_for_plot)
    axes[1, 0].set_ylabel('Savings Rate (%)')
    axes[1, 0].set_title('Average Savings Rate by Quarter', fontweight='bold')
    axes[1, 0].grid(axis='y', alpha=0.3)
    
    for bar, val in zip(bars, savings_pct):
        height = bar.get_height()
        axes[1, 0].text(bar.get_x() + bar.get_width()/2., height,
                       f'{val:.1f}%', ha='center', va='bottom', fontweight='bold')
    
    # Expenditure-to-Income ratio
    ratio_pct = income_expend_df['Expend_Income_Ratio'] * 100
    axes[1, 1].plot(range(len(quarters_for_plot)), ratio_pct, 
                    marker='o', linewidth=2, markersize=10, color='#e67e22')
    axes[1, 1].set_xticks(range(len(quarters_for_plot)))
    axes[1, 1].set_xticklabels(quarters_for_plot)
    axes[1, 1].set_ylabel('Expenditure/Income Ratio (%)')
    axes[1, 1].set_title('Expenditure-to-Income Ratio by Quarter', fontweight='bold')
    axes[1, 1].grid(alpha=0.3)
    
    for x, y in enumerate(ratio_pct):
        axes[1, 1].text(x, y, f'{y:.1f}%', ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(output_dir / '07_viz_income_vs_expenditure_by_quarter.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("✓ Saved: 07_viz_income_vs_expenditure_by_quarter.png")

# ================================================
# SECTION 8: COMPREHENSIVE QUARTERLY SUMMARY
# ================================================

print("\n" + "="*80)
print("SECTION 8: COMPREHENSIVE QUARTERLY SUMMARY")
print("="*80)

# Create master summary table
master_summary = []

for quarter in quarters_2024:
    qdata = quarterly_data[quarter]
    
    summary = {
        'Quarter': quarter_labels[quarter].replace('\n', ' '),
        'Sample_Size': len(qdata),
        'Unique_Consumer_Units': qdata['NEWID'].nunique() if 'NEWID' in qdata.columns else 'N/A',
    }
    
    # Demographics
    if 'AGE_REF' in qdata.columns:
        summary['Avg_Age'] = qdata['AGE_REF'].dropna().mean()
    if 'FAM_SIZE' in qdata.columns:
        summary['Avg_Family_Size'] = qdata['FAM_SIZE'].dropna().mean()
    
    # Income
    if 'FINCBTXM' in qdata.columns:
        income_data = qdata['FINCBTXM'].dropna()
        income_data = income_data[income_data > 0]
        summary['Avg_Income'] = income_data.mean()
        summary['Median_Income'] = income_data.median()
    
    # Expenditure
    if 'TOTEXPPQ' in qdata.columns:
        expend_data = qdata['TOTEXPPQ'].dropna()
        expend_data = expend_data[expend_data > 0]
        summary['Avg_Expenditure'] = expend_data.mean()
        summary['Median_Expenditure'] = expend_data.median()
    
    # Savings rate
    if 'FINCBTXM' in qdata.columns and 'TOTEXPPQ' in qdata.columns:
        analysis_df = qdata[['FINCBTXM', 'TOTEXPPQ']].dropna()
        analysis_df = analysis_df[(analysis_df['FINCBTXM'] > 0) & (analysis_df['TOTEXPPQ'] > 0)]
        if len(analysis_df) > 0:
            savings_rate = 1 - (analysis_df['TOTEXPPQ'].mean() / analysis_df['FINCBTXM'].mean())
            summary['Savings_Rate_%'] = savings_rate * 100
    
    master_summary.append(summary)

master_summary_df = pd.DataFrame(master_summary)
master_summary_df.to_csv(output_dir / '08_quarterly_master_summary.csv', index=False)
print("\n✓ Saved: 08_quarterly_master_summary.csv")

print("\nQUARTERLY MASTER SUMMARY:")
print(master_summary_df.to_string(index=False))

# ====================
# FINAL SUMMARY
# ====================

print("\n" + "="*80)
print("QUARTERLY ANALYSIS COMPLETE!")
print("="*80)

print(f"\nAll outputs saved to: {output_dir}")

print("\nGenerated Files:")
files = sorted(output_dir.glob('*'))
for i, f in enumerate(files, 1):
    print(f"  {i:2d}. {f.name}")

print("\n" + "="*80)
print("KEY QUARTERLY INSIGHTS")
print("="*80)

if len(master_summary_df) > 0:
    print("\nSAMPLE SIZES:")
    for _, row in master_summary_df.iterrows():
        print(f"  {row['Quarter']}: {row['Sample_Size']:,} records")
    
    if 'Avg_Income' in master_summary_df.columns:
        print("\nAVERAGE INCOME TREND:")
        for _, row in master_summary_df.iterrows():
            print(f"  {row['Quarter']}: ${row['Avg_Income']:,.2f}")
        
        income_change = ((master_summary_df['Avg_Income'].iloc[-1] - 
                         master_summary_df['Avg_Income'].iloc[0]) / 
                        master_summary_df['Avg_Income'].iloc[0] * 100)
        print(f"  Change Q2→Q4: {income_change:+.2f}%")
    
    if 'Avg_Expenditure' in master_summary_df.columns:
        print("\nAVERAGE EXPENDITURE TREND:")
        for _, row in master_summary_df.iterrows():
            print(f"  {row['Quarter']}: ${row['Avg_Expenditure']:,.2f}")
        
        expend_change = ((master_summary_df['Avg_Expenditure'].iloc[-1] - 
                         master_summary_df['Avg_Expenditure'].iloc[0]) / 
                        master_summary_df['Avg_Expenditure'].iloc[0] * 100)
        print(f"  Change Q2→Q4: {expend_change:+.2f}%")
    
    if 'Savings_Rate_%' in master_summary_df.columns:
        print("\nSAVINGS RATE TREND:")
        for _, row in master_summary_df.iterrows():
            print(f"  {row['Quarter']}: {row['Savings_Rate_%']:.1f}%")

print("\n" + "="*80)

QUARTERLY ANALYSIS - CONSUMER EXPENDITURE SURVEY 2024
Analyzing: Q2, Q3, Q4 2024
Output directory: data/output-quarterly

SECTION 1: LOADING QUARTERLY DATA
✓ Loaded Q2 2024: 4,673 records, 783 columns
✓ Loaded Q3 2024: 4,612 records, 783 columns
✓ Loaded Q4 2024: 4,601 records, 783 columns

✓ Combined dataset: 13,886 total records

SECTION 2: SAMPLE SIZE ANALYSIS BY QUARTER

Q2 2024 (Apr-Jun):
  Total records: 4,673
  Unique consumer units: 4,673
  Variables: 783

Q3 2024 (Jul-Sep):
  Total records: 4,612
  Unique consumer units: 4,612
  Variables: 783

Q4 2024 (Oct-Dec):
  Total records: 4,601
  Unique consumer units: 4,601
  Variables: 783

✓ Saved: 01_quarterly_sample_sizes.csv
✓ Saved: 01_viz_sample_sizes.png

SECTION 3: DEMOGRAPHIC COMPARISON BY QUARTER

✓ Saved: 02_age_by_quarter.csv

AGE STATISTICS BY QUARTER:
          Quarter  Mean_Age  Median_Age   Std_Dev  Min  Max
Q2 2024 (Apr-Jun) 54.038305        55.0 17.711395   16   88
Q3 2024 (Jul-Sep) 53.820902        54.0 18.009131  

# Quarterly Analysis Summary & Suggested Features

## Executive Summary

This notebook presents a comprehensive quarterly analysis of the Consumer Expenditure Survey (CES) 2024 Interview data, covering Q2, Q3, and Q4 2024. The analysis examines sample sizes, demographics, income patterns, expenditure trends, and the relationship between income and spending across quarters.

### Key Findings

#### 1. Sample Characteristics
- **Total Records**: 13,886 consumer units across three quarters
- **Sample Stability**: Relatively stable sample sizes (~4,600-4,700 per quarter)
- **Data Completeness**: 783 variables per record, providing rich demographic and financial detail

#### 2. Demographics
- **Age Distribution**: Mean age ~54 years across quarters, with consistent distribution
- **Family Size**: Average household size remains stable across quarters
- **Regional Distribution**: Balanced representation across U.S. regions

#### 3. Income Trends
- **Average Income**: ~$106,000-$107,000 per quarter (relatively stable)
- **Median Income**: ~$74,000-$75,000 (indicating right-skewed distribution)
- **Income Stability**: Minimal quarter-to-quarter variation, suggesting stable economic conditions

#### 4. Expenditure Patterns
- **Total Expenditure**: Shows seasonal variations across quarters
- **Category Analysis**: Housing, transportation, and food represent major expenditure categories
- **Spending Trends**: Some categories show quarter-to-quarter fluctuations

#### 5. Income-Expenditure Relationship
- **Correlation**: Strong positive correlation between income and expenditure
- **Savings Rate**: Calculated savings rates vary by quarter
- **Expenditure-to-Income Ratio**: Provides insights into spending behavior relative to income

### Data Quality Notes
- No complete duplicate records detected
- Each consumer unit appears once per quarter (as expected)
- Missing data patterns consistent across quarters

---

## Suggested Features for Modeling

### 1. Demographic Features
- **Age Groups**: Create age bins (e.g., 18-34, 35-49, 50-64, 65+)
- **Family Size Categories**: Small (1-2), Medium (3-4), Large (5+)
- **Regional Indicators**: One-hot encode or label encode region
- **Urban/Rural Classification**: If available in data
- **Housing Tenure**: Owned vs. Rented (if available)

### 2. Income-Based Features
- **Income Quintiles**: Divide into 5 income groups
- **Income-to-Median Ratio**: Relative income position
- **Log Income**: Transform for better distribution properties
- **Income Stability**: Quarter-to-quarter income change (if tracking same households)

### 3. Expenditure Features
- **Total Expenditure**: Aggregate spending per consumer unit
- **Expenditure Categories**: 
  - Food expenditure ratio
  - Housing expenditure ratio
  - Transportation expenditure ratio
  - Healthcare expenditure ratio
  - Entertainment expenditure ratio
- **Expenditure-to-Income Ratio**: Spending relative to income
- **Category Diversity**: Number of categories with non-zero spending
- **Large Purchase Indicators**: Flags for major purchases

### 4. Financial Health Indicators
- **Savings Rate**: (Income - Expenditure) / Income
- **Debt-to-Income Ratio**: If debt data available
- **Expenditure Volatility**: Coefficient of variation across categories
- **Spending Efficiency**: Ratio of essential vs. discretionary spending

### 5. Temporal Features
- **Quarter Indicators**: Q2, Q3, Q4 flags
- **Seasonal Spending Patterns**: Holiday season effects (Q4)
- **Quarter-over-Quarter Changes**: If tracking same households
- **Year-over-Year Comparisons**: If multi-year data available

### 6. Interaction Features
- **Age × Income**: Interaction between age and income level
- **Family Size × Income**: Larger families may have different spending patterns
- **Region × Income**: Regional cost-of-living adjustments
- **Category Interactions**: Relationships between spending categories

### 7. Derived Features
- **Per-Capita Expenditure**: Total expenditure / family size
- **Essential vs. Discretionary**: Ratio of essential spending categories
- **Spending Concentration**: Herfindahl index of category spending
- **Income Elasticity**: Responsiveness of spending to income changes

### 8. Target Variables (for Recommendation System)
- **Next Quarter Spending Prediction**: Predict Q1 2025 spending
- **Category Recommendations**: Which categories to focus on
- **Budget Allocation**: Optimal spending distribution across categories
- **Savings Recommendations**: Target savings rate based on income/demographics

---

## Modeling Recommendations

### For Predictive Modeling:
1. **Feature Engineering**: Create interaction terms and polynomial features
2. **Scaling**: Standardize/normalize numerical features
3. **Handling Skewness**: Apply log transformations to income and expenditure
4. **Categorical Encoding**: Use target encoding or one-hot encoding for categorical variables
5. **Missing Data**: Implement appropriate imputation strategies

### For Recommendation Systems:
1. **Collaborative Filtering**: Find similar consumer units based on spending patterns
2. **Content-Based Filtering**: Use demographic and income features
3. **Hybrid Approach**: Combine both methods for better recommendations
4. **Clustering**: Segment consumers into spending behavior groups
5. **Association Rules**: Identify spending pattern associations

### Model Evaluation:
- Use time-based cross-validation (train on Q2-Q3, test on Q4)
- Consider multiple metrics: RMSE, MAE, R² for regression
- For classification: Precision, Recall, F1-score
- Business metrics: Cost savings, recommendation acceptance rate

---

## Next Steps

1. **Data Integration**: Merge with diary data for transaction-level insights
2. **Feature Engineering**: Implement suggested features above
3. **Model Development**: Build predictive models for spending patterns
4. **Validation**: Test models on holdout quarter
5. **Deployment**: Create recommendation engine for personalized financial advice